In [0]:
dbutils.widgets.text("load_date", "")
load_date = dbutils.widgets.get("load_date")
display(load_date)

In [0]:
%run /Workspace/Repos/junaid20950@gmail.com/human-capital/HC_Project/generic_audit_notebook

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json, ast

# --- ADD: imports for date handling ---
import re
from datetime import datetime
import traceback

CATALOG = "edl_hc_datamart"
SCHEMA = "bronze"
META_CSV = "/Volumes/edl_hc_datamart/config/metadata_config/ingestion_metadata.csv"

# =============================================================================
# CAPTURE JOB CONTEXT (for error logging)
# =============================================================================
try:
    job_context = {
        'job_id': spark.conf.get("spark.databricks.job.id"),
        'job_run_id': spark.conf.get("spark.databricks.job.runId"),
        'cluster_id': spark.conf.get("spark.databricks.clusterUsageTags.clusterId"),
        'notebook_path': dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get(),
        'run_as': spark.sql("SELECT current_user()").collect()[0][0]
    }
    print(f"[JOB CONTEXT] job_id={job_context.get('job_id')}, run_id={job_context.get('job_run_id')}")
except Exception as ctx_err:
    print(f"[WARN] Could not capture job context (running interactively?): {ctx_err}")
    job_context = {}

try:
    # Try to get notebook path even if other context fails
    if "notebook_path" not in job_context or not job_context["notebook_path"]:
        job_context["notebook_path"] = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
except:
    job_context["notebook_path"] = (
        "/Repos/junaid20950@gmail.com/human-capital/HC_Project/(Clone) ingest_jobs_bronze"
    )


# --- ADD: date parsing and parameter retrieval ---
def parse_load_date(value: str) -> str:
    """Validate yyyy-MM-dd and return normalized string."""
    if not value:
        raise ValueError("load_date is empty.")
    if not re.match(r"^\d{4}-\d{2}-\d{2}$", value):
        raise ValueError(f"Invalid load_date '{value}'. Expected format yyyy-MM-dd.")
    datetime.strptime(value, "%Y-%m-%d")
    return value


def get_load_date():
    """
    Try CLI arg --load_date first; fallback to Databricks widgets; default to UTC today.
    Works both in Community Edition notebooks and external Python runners.
    """
    # 1) CLI
    try:
        import argparse

        parser = argparse.ArgumentParser(description="Bronze ingestion with load_date")
        parser.add_argument("--load_date", required=False, help="yyyy-MM-dd")
        args, _ = parser.parse_known_args()
        if args.load_date:
            return parse_load_date(args.load_date)
    except Exception as e:
        print(f"[WARN] argparse failed or not provided: {e}")

    # 2) Widgets
    try:
        dbutils.widgets.text("load_date", "")
        ld = dbutils.widgets.get("load_date")
        if ld:
            return parse_load_date(ld)
    except Exception as e:
        print(f"[WARN] widgets unavailable: {e}")

    # 3) Default
    today = datetime.utcnow().strftime("%Y-%m-%d")
    print(f"[INFO] defaulting load_date to {today} (UTC)")
    return today


# --- ADD: load_date value early for logging & reuse ---
LOAD_DATE = get_load_date()
print(f"[BRONZE] Using load_date: {LOAD_DATE}")


def trim_columns(df):
    return df.select([F.col(c).alias(c.strip()) for c in df.columns])


def parse_options(raw, fmt, landed_path):
    fmt = (fmt or "csv").strip().lower()
    if not raw or raw.strip() == "":
        raw = ""
    s = (
        raw.strip()
        .replace("“", '"')
        .replace("”", '"')
        .replace("‘", "'")
        .replace("’", "'")
    )
    # 1) JSON
    try:
        return json.loads(s)
    except:
        pass
    # 2) Python literal dict
    try:
        lit = ast.literal_eval(s)
        if isinstance(lit, dict):
            return lit
    except:
        pass
    # 3) key=value;key=value
    try:
        opts = {}
        for part in s.split(";"):
            if "=" in part:
                k, v = part.split("=", 1)
                k = k.strip()
                v = v.strip()
                if v.lower() in ("true", "false"):
                    opts[k] = v.lower() == "true"
                else:
                    opts[k] = v.strip('"').strip("'")
        if opts:
            return opts
    except:
        pass
    # 4) defaults
    if fmt == "csv":
        if landed_path and landed_path.lower().endswith(".tsv"):
            return {"header": True, "sep": "\t", "inferSchema": True}
        return {"header": True, "inferSchema": True}
    if fmt == "json":
        return {"multiline": True}
    if fmt == "tsv":
        return {"header": True, "sep": "\t", "inferSchema": True}
    return {}


def source_file_col(df, landed_path: str):
    if "_metadata" in df.columns:
        try:
            _ = df.select(F.col("_metadata.file_path")).limit(1).collect()
            return F.col("_metadata.file_path")
        except:
            pass
    return F.lit(landed_path)


def ingestion_ts_col(ing_str: str):
    norm = F.regexp_replace(F.lit(ing_str or ""), r"T", " ")
    ts = F.try_to_timestamp(norm)
    return F.coalesce(ts, F.current_timestamp())


# -----------------------------------------
# UC catalog/schema
# -----------------------------------------
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE {SCHEMA}")

# ------------------------
# Read metadata
# ------------------------
meta_df = trim_columns(spark.read.option("header", True).csv(META_CSV))

# ---------------------------
# Loop metadata rows
# ---------------------------
failed_sources = []  # Track failures
successful_sources = []  # Track successes

for m in meta_df.collect():
    # =============================================================================
    # WRAP EACH SOURCE IN TRY/EXCEPT FOR GRANULAR ERROR LOGGING
    # =============================================================================
    md = {k: m[k] for k in meta_df.columns}
    source_name = md.get("source_name")
    bronze_table = md.get("bronze_table")
    full_table = f"{CATALOG}.{bronze_table}" if bronze_table else "UNKNOWN"

    # Track start time for this source
    source_start_time = datetime.now()
    audit_run_id = None  # Will be set after successful audit log

    try:
        landed_path = md.get("landed_path")
        source_type = (md.get("source_type") or "file").lower()
        fmt = (md.get("format") or "csv").lower()
        options = parse_options(md.get("options"), fmt, landed_path)

        if not source_name or not bronze_table:
            print(f"Skipping row: missing source_name or table {md}")
            continue

        print(f"\n{'=' * 80}")
        print(f"Processing {source_name} (type={source_type}) -> {full_table}")
        print(f"{'=' * 80}")

        # Only process file sources in this run
        if source_type != "file":
            print(f"Skipping non-file source: {source_type} ({source_name})")
            continue

        if not landed_path:
            raise ValueError(f"Missing landed_path for file source {source_name}")

        # ----- OPTIONAL: if your landed paths are partitioned by date, e.g. /raw/hr/YYYY-MM-DD -----
        # If so, you can derive a dated path:
        # landed_path = f"{landed_path.rstrip('/')}/{LOAD_DATE}"

        # ----- read dataset -----
        print(f"[READ] Reading from {landed_path} (format={fmt})")
        reader = spark.read
        for k, v in options.items():
            reader = reader.option(k, v)

        try:
            df = reader.format(fmt).load(landed_path)
        except Exception as read_err:
            raise Exception(
                f"Failed to read source file: {str(read_err)}"
            ) from read_err

        # Print schema for visibility
        print("[SCHEMA] Input DF:")
        df.printSchema()
        records_read = df.count()
        print(f"[STATS] Records read: {records_read}")

        # -----------------------------------------------------
        # Robust JSON parsing: from_json + schema for 'compensation'
        # -----------------------------------------------------
        if "compensation" in df.columns:
            comp_field = next(
                (f for f in df.schema.fields if f.name == "compensation"), None
            )
            comp_dt = comp_field.dataType if comp_field else None
            print(f"[INFO] compensation datatype: {comp_dt}")

            if isinstance(comp_dt, T.StructType):
                df = (
                    df.withColumn(
                        "salary_amount",
                        F.col("compensation.salary.amount").cast("double"),
                    )
                    .withColumn(
                        "salary_currency", F.col("compensation.salary.currency")
                    )
                    .withColumn(
                        "salary_frequency", F.col("compensation.salary.frequency")
                    )
                    .withColumn(
                        "salary_effective_from",
                        F.to_date(F.col("compensation.salary.effective_from")),
                    )
                    .withColumn(
                        "salary_effective_to",
                        F.to_date(F.col("compensation.salary.effective_to")),
                    )
                    .drop("compensation")
                )
            elif isinstance(comp_dt, T.StringType):
                comp_schema = T.StructType(
                    [
                        T.StructField(
                            "salary",
                            T.StructType(
                                [
                                    T.StructField("effective_to", T.StringType(), True),
                                    T.StructField("amount", T.DoubleType(), True),
                                    T.StructField("currency", T.StringType(), True),
                                    T.StructField(
                                        "effective_from", T.StringType(), True
                                    ),
                                    T.StructField("frequency", T.StringType(), True),
                                ]
                            ),
                            True,
                        )
                    ]
                )
                df = (
                    df.withColumn(
                        "compensation_struct",
                        F.from_json(F.col("compensation"), comp_schema),
                    )
                    .withColumn(
                        "salary_amount", F.col("compensation_struct.salary.amount")
                    )
                    .withColumn(
                        "salary_currency", F.col("compensation_struct.salary.currency")
                    )
                    .withColumn(
                        "salary_frequency",
                        F.col("compensation_struct.salary.frequency"),
                    )
                    .withColumn(
                        "salary_effective_from",
                        F.to_date(F.col("compensation_struct.salary.effective_from")),
                    )
                    .withColumn(
                        "salary_effective_to",
                        F.to_date(F.col("compensation_struct.salary.effective_to")),
                    )
                    .drop("compensation_struct")
                    .drop("compensation")
                )
            else:
                print(
                    f"[WARN] Unexpected compensation type: {comp_dt}. Skipping flattening."
                )

        # ----------------------------
        # Metadata enrichment
        # ----------------------------
        print(f"[TRANSFORM] Enriching with metadata")
        file_path = source_file_col(df, landed_path)
        ing_ts = ingestion_ts_col(md.get("ingestion_ts"))

        df_enriched = (
            df.withColumn("_meta_pipeline_name", F.lit(md.get("pipeline_name")))
            .withColumn("_meta_source_name", F.lit(source_name))
            .withColumn("_meta_batch_id", F.lit(md.get("batch_id")))
            .withColumn("_meta_run_id", F.lit(md.get("run_id")))
            .withColumn("_meta_schema_version", F.lit(md.get("schema_version")))
            .withColumn("_meta_producer_system", F.lit(md.get("producer_system")))
            .withColumn("_meta_ingestion_user", F.lit(md.get("ingestion_user")))
            .withColumn("_meta_ingestion_ts", ing_ts)
            .withColumn("_meta_source_file", file_path)
            .withColumn("_meta_load_time", F.current_timestamp())
            .withColumn("_meta_load_date", F.lit(LOAD_DATE))  # <-- ADDED
        ).drop("compensation")

        # ----------------------------
        # Write to UC table (overwrite as per your original)
        # ----------------------------
        print(f"[WRITE] Writing to {full_table}")
        try:
            (
                df_enriched.write.mode("overwrite")
                .format("delta")
                .saveAsTable(full_table)
            )
        except Exception as write_err:
            raise Exception(
                f"Failed to write to table {full_table}: {str(write_err)}"
            ) from write_err

        # Count records for audit
        records_written = df_enriched.count()
        print(f"[STATS] Records written: {records_written}")

        # Calculate duration
        source_end_time = datetime.now()
        duration = int((source_end_time - source_start_time).total_seconds())

        # Log audit (SUCCESS)
        print(f"[AUDIT] Logging successful ingestion")
        run_id = md.get("run_id")
        log_audit_ingestion(
            pipeline_name="ingest_jobs_bronze",
            layer="bronze",
            source_name=source_name,
            target_table=full_table,
            source_type=source_type,
            batch_id=md.get("batch_id"),
            run_id=run_id,
            trigger_type=job_context.get("job_id") and "scheduled" or "manual",
            run_start_ts=source_start_time,
            run_end_ts=source_end_time,
            last_status="SUCCESS",
            records_read=records_read,
            records_written=records_written,
            schema_version_applied=md.get("schema_version"),
            producer_system=md.get("producer_system"),
            ingestion_user=md.get("ingestion_user"),
            last_load_date=LOAD_DATE,
        )
        audit_run_id = run_id  # Store for potential error linking

        print(
            f"✓ SUCCESS: {source_name} -> {full_table} | {records_read} → {records_written} records | {duration}s"
        )
        successful_sources.append(source_name)

    # =============================================================================
    # ERROR HANDLING: Catch and log any errors for this source
    # =============================================================================
    except Exception as e:
        source_end_time = datetime.now()
        duration = int((source_end_time - source_start_time).total_seconds())
        error_msg = str(e)

        # Classify error type
        error_type = "UNKNOWN"
        if "DELTA_METADATA_MISMATCH" in error_msg or "schema" in error_msg.lower():
            error_type = "SCHEMA_MISMATCH"
        elif "PERMISSION_DENIED" in error_msg or "permission" in error_msg.lower():
            error_type = "PERMISSION_DENIED"
        elif "TABLE_OR_VIEW_NOT_FOUND" in error_msg or "not found" in error_msg.lower():
            error_type = "TABLE_NOT_FOUND"
        elif "FileNotFoundException" in error_msg or "Path does not exist" in error_msg:
            error_type = "FILE_NOT_FOUND"
        elif "Failed to read" in error_msg:
            error_type = "READ_ERROR"
        elif "Failed to write" in error_msg:
            error_type = "WRITE_ERROR"
        elif "Missing landed_path" in error_msg or "ValueError" in str(type(e)):
            error_type = "CONFIG_ERROR"

        print(f"\n{'=' * 80}")
        print(f"✗ ERROR in {source_name}: {error_type}")
        print(f"{'=' * 80}")
        print(f"Message: {error_msg}")

        # Log the error with full context
        try:
            error_id = log_pipeline_error(
                pipeline_name="ingest_jobs_bronze",
                layer="bronze",
                source_name=source_name,
                target_table=full_table,
                error_message=error_msg,
                exception_obj=e,
                error_type=error_type,
                # Job context
                job_id=job_context.get("job_id"),
                job_run_id=job_context.get("job_run_id"),
                cluster_id=job_context.get("cluster_id"),
                run_as=job_context.get("run_as"),
                launched_by=job_context.get("job_id") and "scheduled" or "manual",
                notebook_path=job_context.get("notebook_path"),
                # Timing
                job_started_at=source_start_time,
                job_ended_at=source_end_time,
                duration_seconds=duration,
                # Linking
                audit_run_id=audit_run_id,
                batch_id=md.get("batch_id"),
                # Additional context
                notes=f"Source file: {landed_path}, Format: {fmt}, Load date: {LOAD_DATE}",
            )
            print(f"✓ Error logged with error_id: {error_id}")
        except Exception as log_err:
            print(f"[WARN] Failed to log error to error_log table: {log_err}")
            print(f"[WARN] Original error: {error_msg}")

        # Track failed source
        failed_sources.append((source_name, error_type, error_msg))

        # Continue to next source instead of failing entire job
        print(f"[INFO] Continuing to next source...\n")
        continue

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print(f"\n\n{'=' * 80}")
print(f"BRONZE INGESTION COMPLETE")
print(f"{'=' * 80}")
print(f"Load Date: {LOAD_DATE}")
print(f"Successful sources: {len(successful_sources)}")
print(f"Failed sources: {len(failed_sources)}")

if successful_sources:
    print(f"\n✓ Successfully processed:")
    for src in successful_sources:
        print(f"  - {src}")

if failed_sources:
    print(f"\n✗ Failed sources:")
    for src, err_type, err_msg in failed_sources:
        print(f"  - {src} ({err_type}): {err_msg[:100]}...")
    print(f"\nCheck error_log table for details:")
    print(
        f"  SELECT * FROM edl_hc_datamart.audit.error_log WHERE pipeline_name = 'ingest_jobs_bronze' ORDER BY error_timestamp DESC"
    )

print(f"\nAudit log:")
print(
    f"  SELECT * FROM edl_hc_datamart.audit.audit_ingestion WHERE pipeline_name = 'ingest_jobs_bronze' ORDER BY created_at DESC"
)
print(f"{'=' * 80}")

# Raise error if ALL sources failed (but not if some succeeded)
if failed_sources and not successful_sources:
    raise Exception(
        f"All {len(failed_sources)} source(s) failed to process. Check error_log table for details."
    )